# Урок 25. Базы данных: зачем нужны и как устроены

8 класс · III четверть

[⬆ Все уроки](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-08/index.ipynb) · [← Урок 24](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-08/urok-24.ipynb) · [Урок 26 →](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-08/urok-26.ipynb)

---

Чем база данных лучше десятка таблиц в папке. Запись, поле, ключ. Связь между таблицами. Как выглядит запрос на выборку. Практика в Access: конструктор таблиц, схема данных, первые запросы.

In [ ]:
#@title 🚀 Регистрация { display-mode: "form" }
#@markdown Впиши свои данные и запусти ячейку (Shift+Enter).
#@markdown Если заводил секрет `INFORMATIKA` и включил к нему доступ
#@markdown (🔑 на панели слева) — поля можно оставить пустыми.
#@markdown
#@markdown Colab спросит разрешение на доступ к аккаунту — нажми
#@markdown «Разрешить». Так проверка понимает, чья это работа.
ФИО = "" #@param {type:"string"}
#@markdown Класс — с буквой, например 8А
Класс = "" #@param {type:"string"}

import importlib, urllib.request
for адрес in ("https://raw.githubusercontent.com/LeksssGray/informatika-7-11/main/lib/schoolinf.py",
              "https://cdn.jsdelivr.net/gh/LeksssGray/informatika-7-11@main/lib/schoolinf.py"):
    try:
        urllib.request.urlretrieve(адрес, "schoolinf.py")
        break
    except Exception as ошибка:
        print("⚠️ Не скачалась библиотека проверки:", ошибка)
else:
    raise SystemExit("Проверь интернет и запусти эту ячейку ещё раз. "
                     "Не помогло — покажи это сообщение учителю.")
import schoolinf as si
importlib.reload(si)
si.start(lesson="08-25", name=ФИО, klass=Класс)

## Разбираемся

### Когда таблицы перестают справляться

Школьная библиотека ведёт учёт в таблице: книга, автор, кто взял,
когда вернуть. Пока книг сотня, всё хорошо. Потом начинается:

* Иванов взял три книги — его фамилия записана трижды, и в одной
  строке она «Иванов И.», в другой «иванов», в третьей «Иванов Иван»;
* библиотекарь исправил фамилию в одной строке и забыл в двух;
* файл открыли на двух компьютерах и сохранили по очереди —
  половина записей пропала;
* нужно найти всех должников по седьмым классам, а фильтры дают
  не то.

Все эти беды — про одно: **данные дублируются и разъезжаются**.
Лечит их база данных.

> **База данных** — организованное хранилище данных, где сведения
> разложены по связанным таблицам и не повторяются.

### Из чего состоит таблица базы

| Термин | Что это | В школьном журнале |
|---|---|---|
| **Поле** | столбец, один признак | «фамилия», «класс» |
| **Запись** | строка, один объект | один ученик целиком |
| **Тип поля** | что в нём хранится | число, текст, дата |
| **Ключ** | поле, по которому запись узнаётся однозначно | номер читательского билета |

Слова стоит запомнить: на экзамене спрашивают именно их, а не
«строка» и «столбец».

### Ключ

**Ключ** — это поле (или несколько), которое не повторяется
ни у одной записи. Фамилия ключом быть не может: Ивановых в школе
двое. Номер билета, ISBN книги, номер паспорта — могут.

Если подходящего поля нет, его просто заводят: колонка `id`
с числами 1, 2, 3… Так устроено большинство реальных баз.

### Связь между таблицами

Главная идея: вместо одной большой таблицы делают несколько
маленьких и связывают их ключами.

```
Читатели                     Выдачи
┌────┬───────────┬───────┐   ┌────┬──────────┬────────┐
│ id │ фамилия   │ класс │   │ id │ читатель │ книга  │
├────┼───────────┼───────┤   ├────┼──────────┼────────┤
│  1 │ Иванов    │ 8А    │   │  1 │    1     │  12    │
│  2 │ Петрова   │ 8Б    │   │  2 │    1     │  45    │
└────┴───────────┴───────┘   │  3 │    2     │  12    │
                             └────┴──────────┴────────┘
```

В таблице выдач фамилии нет — стоит только номер читателя. Поэтому:

* фамилия хранится **в одном месте**, исправляется один раз
  и сразу везде;
* опечатки невозможны — номер либо есть, либо нет;
* места нужно меньше.

Поле, которое ссылается на ключ другой таблицы, называется
**внешним ключом**.

### Запрос

С базой разговаривают **запросами** на языке **SQL**. В Access
запрос можно собрать и мышкой, в конструкторе, — но внутри это
всё равно SQL. Выглядит он почти как английская фраза:

```sql
SELECT фамилия, класс      -- какие поля показать
FROM читатели              -- из какой таблицы
WHERE класс = '8А'         -- какие записи взять
ORDER BY фамилия;          -- как отсортировать
```

Четыре слова закрывают едва ли не половину всех запросов на свете:

| Слово | Смысл |
|---|---|
| `SELECT` | что показать |
| `FROM` | откуда взять |
| `WHERE` | условие отбора |
| `ORDER BY` | сортировка |

Заметьте: вы описываете **что** нужно, а не **как** это найти.
Перебирать записи, сравнивать и сортировать база будет сама —
этим SQL и отличается от Python.

### Почему база ищет быстро

Потому что по ключевым полям она хранит **индекс** — отсортированный
справочник, по которому можно искать делением пополам, как слово
в бумажном словаре: открыл посередине, отбросил половину, снова
посередине.

Миллион записей — двадцать шагов вместо миллиона. Сам этот алгоритм
мы напишем на следующем уроке.

## Практическая работа. Библиотека в Access

Базы данных делают в **Microsoft Access** — он входит в тот же
пакет Office, что Word и Excel. Сегодня соберём базу школьной
библиотеки из двух связанных таблиц и зададим ей первые вопросы.

Если что-то не успеете, [готовая база с урока](https://raw.githubusercontent.com/LeksssGray/informatika-7-11/main/files/access/g08-25-biblioteka.accdb)
поможет догнать — но сначала попробуйте сами.

### Шаг 1. Новая база

Запустите Access. На стартовом экране выберите **Пустая база
данных**.

<img src="https://raw.githubusercontent.com/LeksssGray/informatika-7-11/main/img/access/g08-25-start.png" width="900" alt="Стартовый экран Access">

*Стартовый экран Access*

Справа внизу появится поле **Имя файла**. Впишите `Библиотека`,
нажмите на папку рядом и выберите, куда сохранить (например,
«Документы»). Нажмите **Создать**.

> В отличие от Word и Excel, Access сохраняет базу **сразу**,
> при создании, и дальше записывает каждое изменение сам. Кнопки
> «Сохранить всё» здесь нет — и не нужна.

Если сверху появится жёлтая полоса **Предупреждение системы
безопасности** — нажмите **Включить содержимое**: это ваша база,
ей можно доверять.

### Шаг 2. Таблица «Читатели» в конструкторе

Access сразу открыл пустую «Таблицу1». Закройте её крестиком
справа — таблицу будем делать правильно, в конструкторе:
**Создание → Конструктор таблиц**.

<img src="https://raw.githubusercontent.com/LeksssGray/informatika-7-11/main/img/access/g08-25-sozdanie.png" width="900" alt="Вкладка «Создание»">

*Вкладка «Создание»*

В конструкторе у каждого поля три столбца: **имя**, **тип данных**
и описание (необязательно). Заполните:

| Имя поля | Тип данных | Зачем |
|---|---|---|
| Код | Счетчик | номер читателя; Access проставит сам: 1, 2, 3… |
| Фамилия | Короткий текст | |
| Имя | Короткий текст | |
| Класс | Короткий текст | «8А» — это текст, а не число |

Тип выбирают из выпадающего списка — щёлкните в столбце «Тип
данных» и нажмите на стрелочку.

Теперь **ключ**: щёлкните по строке «Код» и нажмите **Ключевое
поле** на ленте. Слева от строки появится ключик.

<img src="https://raw.githubusercontent.com/LeksssGray/informatika-7-11/main/img/access/g08-25-konstruktor.png" width="900" alt="Конструктор таблицы «Читатели»">

*Конструктор таблицы «Читатели»*

Сохраните таблицу: `Ctrl` + `S` → имя `Читатели` → **ОК**.

### Шаг 3. Заполняем

Переключитесь в режим таблицы: **Главная → Режим** (значок
таблицы слева на ленте) или двойной щелчок по «Читатели» в списке
слева. Впишите девять читателей. Поле «Код» не трогайте —
Access заполнит его сам.

<img src="https://raw.githubusercontent.com/LeksssGray/informatika-7-11/main/img/access/g08-25-chitateli.png" width="900" alt="Таблица «Читатели» с данными">

*Таблица «Читатели» с данными*

| Фамилия | Имя | Класс |
|---|---|---|
| Иванов | Пётр | 8А |
| Петрова | Анна | 8Б |
| Сидоров | Максим | 8А |
| Кузнецова | Мария | 8А |
| Орлов | Денис | 8Б |
| Никитина | Ольга | 8Б |
| Жуков | Илья | 8А |
| Смирнова | Ева | 8Б |
| Волков | Артём | 8А |

Запись сохраняется, как только вы уходите со строки. Отдельно
сохранять ничего не нужно.

### Шаг 4. Таблица «Выдачи»

Снова **Создание → Конструктор таблиц**:

| Имя поля | Тип данных |
|---|---|
| Код | Счетчик (ключевое поле) |
| Читатель | Числовой |
| Книга | Короткий текст |
| Дата | Дата и время |

<img src="https://raw.githubusercontent.com/LeksssGray/informatika-7-11/main/img/access/g08-25-vydachi-konstruktor.png" width="900" alt="Конструктор таблицы «Выдачи»">

*Конструктор таблицы «Выдачи»*

Поле **Читатель** — это **внешний ключ**: в нём будет не фамилия,
а код читателя из первой таблицы. Поэтому тип у него **Числовой** —
такой же, как у счётчика.

Сохраните как `Выдачи`. Пока не заполняйте — сначала свяжем
таблицы.

### Шаг 5. Схема данных

**Работа с базами данных → Схема данных**.

<img src="https://raw.githubusercontent.com/LeksssGray/informatika-7-11/main/img/access/g08-25-rabota-s-bazami.png" width="900" alt="Кнопка «Схема данных»">

*Кнопка «Схема данных»*

Если открылось окно **Добавление таблицы** — выделите обе таблицы
и нажмите **Добавить**, потом **Закрыть**. Если окна нет — нажмите
**Добавить таблицы** на ленте.

Теперь **перетащите мышью** поле «Код» из «Читатели» на поле
«Читатель» в «Выдачи». Откроется окно **Изменение связей**:

* поставьте галочку **Обеспечение целостности данных** —
  тогда Access не даст записать выдачу несуществующему читателю;
* нажмите **Создать**.

<img src="https://raw.githubusercontent.com/LeksssGray/informatika-7-11/main/img/access/g08-25-shema.png" width="900" alt="Схема данных: связь «один ко многим»">

*Схема данных: связь «один ко многим»*

На линии связи появятся **1** и **∞**: один читатель — много выдач.
Закройте схему и согласитесь сохранить макет.

### Шаг 6. Заполняем выдачи

Откройте «Выдачи» и впишите двенадцать строк. В поле «Читатель» —
**код** из таблицы «Читатели» (Иванов — 1, Петрова — 2 и так далее).

| Читатель | Книга | Дата |
|---|---|---|
| 1 | Капитанская дочка | 02.09.2026 |
| 1 | Недоросль | 09.09.2026 |
| 2 | Капитанская дочка | 03.09.2026 |
| 3 | Ревизор | 04.09.2026 |
| 1 | Ревизор | 16.09.2026 |
| 4 | Мцыри | 05.09.2026 |
| 5 | Капитанская дочка | 08.09.2026 |
| 6 | Мцыри | 10.09.2026 |
| 7 | Недоросль | 11.09.2026 |
| 2 | Ревизор | 12.09.2026 |
| 9 | Мцыри | 15.09.2026 |
| 4 | Ревизор | 17.09.2026 |

Попробуйте вписать читателя `99`. Access откажется: такой записи
в «Читатели» нет — это и есть целостность данных.

### Шаг 7. Первый запрос — в конструкторе

Вопрос: **кто из читателей учится в 8А**, по алфавиту?

**Создание → Конструктор запросов**. В окне **Добавление таблицы**
выберите «Читатели» → **Добавить** → **Закрыть**.

Внизу — бланк запроса. Перетащите в него поля «Фамилия», «Имя»,
«Класс» (или дважды щёлкните по ним). Затем:

* в столбце «Класс», строка **Условие отбора**: `8А`
  (кавычки Access поставит сам);
* в столбце «Фамилия», строка **Сортировка**: *по возрастанию*.

<img src="https://raw.githubusercontent.com/LeksssGray/informatika-7-11/main/img/access/g08-25-zapros-konstruktor.png" width="900" alt="Запрос в конструкторе">

*Запрос в конструкторе*

Нажмите **Выполнить** (красный восклицательный знак на ленте).

<img src="https://raw.githubusercontent.com/LeksssGray/informatika-7-11/main/img/access/g08-25-zapros-rezultat.png" width="900" alt="Результат запроса">

*Результат запроса*

Сохраните запрос: `Ctrl` + `S` → `Читатели 8А`.

### Шаг 8. Тот же запрос на SQL

Конструктор — это просто удобная обёртка над SQL. Посмотрите,
что он написал: **Главная → Режим → Режим SQL**.

<img src="https://raw.githubusercontent.com/LeksssGray/informatika-7-11/main/img/access/g08-25-zapros-sql.png" width="900" alt="Режим SQL">

*Режим SQL*

Те самые `SELECT`, `FROM`, `WHERE`, `ORDER BY` из теории. Access
добавляет лишние скобки и имя таблицы перед каждым полем — так
ему надёжнее, но смысл тот же.

### Шаг 9. Запрос по двум таблицам

Вопрос: **кто какую книгу взял**?

Новый запрос в конструкторе, добавьте **обе** таблицы. Access сам
нарисует между ними линию связи — ту, что вы сделали на схеме.
В бланк: «Фамилия» из «Читатели», «Книга» и «Дата» из «Выдачи».
Сортировка по фамилии.

<img src="https://raw.githubusercontent.com/LeksssGray/informatika-7-11/main/img/access/g08-25-dve-tablicy.png" width="900" alt="Запрос по двум таблицам">

*Запрос по двум таблицам*

<img src="https://raw.githubusercontent.com/LeksssGray/informatika-7-11/main/img/access/g08-25-dve-tablicy-rezultat.png" width="900" alt="Кто что взял">

*Кто что взял*

В таблице «Выдачи» фамилий нет — только номера. Фамилию запрос
подтянул по связи. Сохраните как `Кто что взял`.

## Проверяем себя

### Задача 1. Как называется столбец

Как называется столбец таблицы базы данных?

In [ ]:
#@title 🧩 Задача 1. Столбец таблицы { display-mode: "form" }
#@markdown Выбери термин
столбец = "выбери ответ" #@param ["выбери ответ", "поле", "запись", "ключ", "индекс"]

if "si" not in globals(): raise SystemExit("⚠️ Сначала запусти ячейку «🚀 Регистрация» в начале урока (Shift+Enter) и дождись, пока она закончит работу.")
si.ответ("1", столбец, "637a51fbbacce29d",
         hint="Один признак у всех объектов сразу.")

### Задача 2. Как называется строка

Как называется строка таблицы — сведения об одном объекте целиком?

In [ ]:
#@title 🧩 Задача 2. Строка таблицы { display-mode: "form" }
#@markdown Выбери термин
строка_таблицы = "выбери ответ" #@param ["выбери ответ", "поле", "запись", "ключ", "индекс"]

if "si" not in globals(): raise SystemExit("⚠️ Сначала запусти ячейку «🚀 Регистрация» в начале урока (Shift+Enter) и дождись, пока она закончит работу.")
si.ответ("2", строка_таблицы, "1fa00777a7ee735a",
         hint="Одна книга, один ученик, один читатель.")

### Задача 3. Поле, которое не повторяется

Как называется поле, по которому запись узнаётся однозначно?

In [ ]:
#@title 🧩 Задача 3. Однозначное поле { display-mode: "form" }
#@markdown Выбери термин
однозначное = "выбери ответ" #@param ["выбери ответ", "поле", "запись", "ключ", "тип"]

if "si" not in globals(): raise SystemExit("⚠️ Сначала запусти ячейку «🚀 Регистрация» в начале урока (Shift+Enter) и дождись, пока она закончит работу.")
si.ответ("3", однозначное, "b3db64debdf2314c",
         hint="Номер читательского билета, ISBN, номер паспорта.")

### Задача 4. Сколько в 8А

Сколько записей вернул запрос «Читатели 8А»? Число видно внизу
окна: «Запись: 1 из …».

In [ ]:
#@title 🧩 Задача 4. Читатели 8А { display-mode: "form" }
#@markdown Впиши число
в_8а = 0 #@param {type:"integer"}

if "si" not in globals(): raise SystemExit("⚠️ Сначала запусти ячейку «🚀 Регистрация» в начале урока (Shift+Enter) и дождись, пока она закончит работу.")
si.ответ("4", в_8а, "ef2d127de37b942b",
         hint="Условие отбора — 8А в столбце «Класс».")

### Задача 5. Сколько книг у Иванова

Сколько раз Иванов брал книги? Добавьте в запрос «Кто что взял»
условие отбора `Иванов` в столбце «Фамилия» и выполните.

In [ ]:
#@title 🧩 Задача 5. Выдачи Иванова { display-mode: "form" }
#@markdown Впиши число
у_иванова = 0 #@param {type:"integer"}

if "si" not in globals(): raise SystemExit("⚠️ Сначала запусти ячейку «🚀 Регистрация» в начале урока (Shift+Enter) и дождись, пока она закончит работу.")
si.ответ("5", у_иванова, "4e07408562bedb8b",
         hint="Условие — в столбце «Фамилия», не «Книга».")

### Задача 6. Слово запроса

С какого слова начинается запрос на выборку данных?
Запишите заглавными буквами.

In [ ]:
#@title 🧩 Задача 6. Начало запроса { display-mode: "form" }
#@markdown Впиши слово
слово = "" #@param {type:"string"}

if "si" not in globals(): raise SystemExit("⚠️ Сначала запусти ячейку «🚀 Регистрация» в начале урока (Shift+Enter) и дождись, пока она закончит работу.")
si.ответ("6", слово, "8dac625925dff4ce",
         hint="По-английски «выбрать».")

## Домашнее задание

### 1. Своя база в Access

Сделайте в Access базу на свою тему: фильмотека, коллекция игр,
домашние питомцы, спортивная секция, рецепты — что угодно, где
есть **два вида объектов** и связь «один ко многим» между ними.
Например: режиссёры и фильмы, игроки и матчи, авторы и рецепты.

Требования:

1. **две таблицы**, созданные в **конструкторе**; в каждой
   не меньше трёх полей;
2. у каждой таблицы ключевое поле — счётчик;
3. хотя бы три **разных** типа данных: текст, число, дата или
   логический (да/нет);
4. во второй таблице — числовое поле-ссылка на первую;
5. связь на **схеме данных** с галочкой «Обеспечение целостности
   данных»;
6. в первой таблице **не меньше 5** записей, во второй —
   **не меньше 10**;
7. **два запроса**, сохранённых с понятными именами: один
   с условием отбора и сортировкой по одной таблице, второй —
   по обеим таблицам сразу.

Проверьте себя: если в первой таблице исправить название, сколько
ячеек придётся поменять? Правильный ответ — одну.

Сдайте файл `.accdb` учителю. Перед отправкой закройте Access —
иначе рядом останется служебный файл `.laccdb`, его отправлять
не нужно.

### 2. Сколько в 8Б

Скопируйте запрос «Читатели 8А» (правой кнопкой в списке слева →
**Копировать**, потом **Вставить**, имя `Читатели 8Б`) и поменяйте
условие отбора. Сколько записей он вернёт?

In [ ]:
#@title 🏠 Домашнее 2. Читатели 8Б { display-mode: "form" }
#@markdown Впиши число
в_8б = 0 #@param {type:"integer"}

if "si" not in globals(): raise SystemExit("⚠️ Сначала запусти ячейку «🚀 Регистрация» в начале урока (Shift+Enter) и дождись, пока она закончит работу.")
si.ответ("дз2", в_8б, "4b227777d4dd1fc6",
         hint="Открыть копию в конструкторе, поменять 8А на 8Б.")

### 3. Сколько записей вернёт запрос

В таблице `читатели` четыре записи. Сколько строк вернёт запрос
`SELECT COUNT(*) FROM читатели`?

Осторожно: вопрос не про количество записей, а про количество
строк **в ответе**.

In [ ]:
#@title 🏠 Домашнее 3. Строк в ответе { display-mode: "form" }
#@markdown Впиши число
строк = 0 #@param {type:"integer"}

if "si" not in globals(): raise SystemExit("⚠️ Сначала запусти ячейку «🚀 Регистрация» в начале урока (Shift+Enter) и дождись, пока она закончит работу.")
si.ответ("дз3", строк, "6b86b273ff34fce1",
         hint="COUNT возвращает одно число — значит, одну строку с одним полем.")

---

### Любопытно

Реляционную модель — ту самую, с таблицами и ключами — придумал
в 1970 году математик Эдгар Кодд, сотрудник IBM. Начальство идею
встретило прохладно: у компании уже была своя база данных, и продавать
её было выгоднее.

Коддовскую статью прочитали в маленькой компании в Калифорнии
и сделали по ней продукт. Компания называлась Oracle и стала одной
из крупнейших в мире. А SQL, придуманный в тех же лабораториях IBM,
работает сегодня примерно везде — от банкомата до вашего телефона,
где на этом же sqlite3 хранятся переписки и заметки.

---

## Отчёт по уроку

Запусти ячейку ниже, когда решишь задачи. Результат уйдёт учителю автоматически.

In [ ]:
si.report()

---

[← Урок 24](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-08/urok-24.ipynb) · [⬆ Все уроки](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-08/index.ipynb) · [🏠 Ко всем классам](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/index.ipynb) · [Урок 26 →](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-08/urok-26.ipynb)